# 0001 - Two Sum

Neste notebook, vamos resolver o problema **Two Sum** de um jeito didático, começando pela solução mais simples e evoluindo até a abordagem ótima.

A ideia é que este material possa ser lido sozinho, estudado com calma e depois exportado para um **Jupyter Book** com boa apresentação no GitHub Pages.


## Enunciado

### Enunciado original

Enunciado original

### Enunciado traduzido em linguagem simples

O que o problema quer é encontrar **duas posições diferentes** do vetor em que os valores somem exatamente o `target`.
Se `nums[i] + nums[j] == target`, então devemos devolver os índices `i` e `j`.


## Imports

Colocar aqui todos os imports necessários


In [ ]:
from __future__ import annotations

from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")


def plot_growth(title: str, x: pd.Series | np.ndarray, curves: dict[str, pd.Series | np.ndarray]) -> None:
    '''Plota curvas de crescimento para fins didáticos.

    Parâmetros:
    ----------
    title : str
        Título do gráfico.
    x : pd.Series | np.ndarray
        Valores do eixo X.
    curves : dict[str, pd.Series | np.ndarray]
        Dicionário com o nome de cada curva e os valores do eixo Y.
    '''
    fig, ax = plt.subplots(figsize=(10, 5))
    for label, y in curves.items():
        sns.lineplot(x=x, y=y, ax=ax, label=label, linewidth=2.5)

    ax.set_title(title)
    ax.set_xlabel("Tamanho da entrada (n)")
    ax.set_ylabel("Crescimento relativo")
    ax.legend()
    plt.tight_layout()
    plt.show()


def validar_resposta_two_sum(nums: list[int], target: int, resposta: list[int]) -> bool:
    '''Verifica se a resposta retornada resolve corretamente o problema.

    Parâmetros:
    ----------
    nums : list[int]
        Lista original de números.
    target : int
        Soma desejada.
    resposta : list[int]
        Lista com dois índices supostamente válidos.

    Retorno:
    -------
    bool
        `True` quando a resposta é válida e `False` caso contrário.
    '''
    if not isinstance(resposta, list):
        return False
    if len(resposta) != 2:
        return False

    i, j = resposta
    if not isinstance(i, int) or not isinstance(j, int):
        return False
    if i == j:
        return False
    if not (0 <= i < len(nums) and 0 <= j < len(nums)):
        return False

    return nums[i] + nums[j] == target


def gerar_caso_benchmark(tamanho: int) -> tuple[list[int], int]:
    '''Gera um caso de benchmark com solução única e no fim da lista.

    Parâmetros:
    ----------
    tamanho : int
        Número de elementos da lista.

    Retorno:
    -------
    tuple[list[int], int]
        A lista de entrada e o valor de `target`.

    Exceções:
    --------
    ValueError
        Levantada quando `tamanho < 2`.

    Exemplos de uso:
    ----------------
    >>> nums, target = gerar_caso_benchmark(5)
    >>> nums
    [1, 2, 3, 4, 5]
    >>> target
    9
    '''
    if tamanho < 2:
        raise ValueError("O tamanho mínimo do benchmark é 2.")

    nums = list(range(1, tamanho + 1))
    target = nums[-1] + nums[-2]
    return nums, target


def medir_media_execucao(funcao, nums: list[int], target: int, repeticoes: int = 7) -> float:
    '''Mede o tempo médio de execução de uma função de Two Sum.

    Parâmetros:
    ----------
    funcao : callable
        Função a ser avaliada.
    nums : list[int]
        Lista de entrada.
    target : int
        Soma desejada.
    repeticoes : int, default = 7
        Quantas vezes repetir a medição.

    Retorno:
    -------
    float
        Tempo médio de execução, em segundos.

    Exceções:
    --------
    AssertionError
        Levantada se a função não retornar uma resposta válida.
    '''
    tempos: list[float] = []
    for _ in range(repeticoes):
        inicio = perf_counter()
        resposta = funcao(nums, target)
        fim = perf_counter()
        assert validar_resposta_two_sum(nums, target, resposta)
        tempos.append(fim - inicio)

    return float(np.mean(tempos))


def executar_benchmark(funcao, nome_solucao: str, tamanhos: list[int], repeticoes: int = 7) -> pd.DataFrame:
    '''Executa benchmark em vários tamanhos de entrada.

    Parâmetros:
    ----------
    funcao : callable
        Implementação de Two Sum a ser testada.
    nome_solucao : str
        Nome amigável da solução.
    tamanhos : list[int]
        Tamanhos de entrada que serão testados.
    repeticoes : int, default = 7
        Quantas repetições usar para cada tamanho.

    Retorno:
    -------
    pd.DataFrame
        Tabela com tamanho da entrada, solução e tempo médio observado.
    '''
    registros = []
    for tamanho in tamanhos:
        nums, target = gerar_caso_benchmark(tamanho)
        tempo_medio = medir_media_execucao(funcao, nums, target, repeticoes=repeticoes)
        registros.append(
            {
                "tamanho_entrada": tamanho,
                "solucao": nome_solucao,
                "tempo_execucao_s": tempo_medio,
                "tempo_execucao_ms": tempo_medio * 1000,
            }
        )

    return pd.DataFrame(registros)


## Funções uteis

**Imports**

Colocar aqui todos os imports necessários

**Funções uteis**

**Intuição geral**

Vamos construir a solução em três etapas:

1. **Força bruta**: testar todos os pares possíveis.
2. **Melhoria intermediária**: continuar testando pares, mas evitando verificações redundantes.
3. **Solução ótima**: usar um `hash map` para descobrir o complemento em tempo constante amortizado.

Essa progressão é ótima para estudo porque mostra como pensar sobre o problema, como reduzir trabalho desnecessário e como chegar à solução mais eficiente.

**Preparação do ambiente**

Este notebook usa principalmente:

- `matplotlib`
- `seaborn`
- `pandas`
- `numpy`

Se você quiser instalar tudo com `uv`, uma opção prática é:

```bash
uv add matplotlib seaborn pandas notebook
```

As dependências já fazem sentido neste projeto porque o arquivo `pyproject.toml` também inclui `jupyterlab`, `matplotlib`, `numpy`, `pandas` e `seaborn`.


## Solução 1 - Força bruta


### Descrição da solução

**Descrição da solução**

A primeira ideia é a mais direta possível: testar todos os pares de posições `(i, j)` e verificar se a soma dá `target`.

Essa solução é valiosa porque é fácil de pensar e fácil de implementar. Em entrevista, ela costuma aparecer como ponto de partida para depois ser melhorada.

**Como chegamos ao `O(n^2)`?**

Vamos olhar para o trabalho feito pelos laços.

- O laço externo escolhe cada posição da lista: isso acontece cerca de `n` vezes.
- Para cada posição, o laço interno percorre a lista de novo: mais cerca de `n` comparações.
- No pior caso, sem saída antecipada, fazemos aproximadamente `n * (n - 1)` somas/comparações.

Em linguagem mais formal, o custo é proporcional ao número de pares testados. Como `n * (n - 1)` é um polinômio de grau 2, desprezamos constantes e termos menores e ficamos com `O(n^2)`.

**Por que o espaço é `O(1)`?**

A solução usa apenas algumas variáveis auxiliares para os índices e as comparações. Não criamos uma estrutura proporcional ao tamanho da entrada.

**Implementação da força bruta**

Vamos verificar se a implementação funciona nos três exemplos do enunciado, em um caso pequeno extra e em um caso com números negativos.


### Ordem de complexidade (memória e processamento)

**Ordem de complexidade (memória e processamento)**

- **Tempo:** `O(n^2)`
- **Espaço extra:** `O(1)`


### Mini walkthrough

**Intuição**

- Pegamos um elemento de `nums`.
- Comparamos esse elemento com todos os outros.
- Se a soma dos dois for igual ao `target`, encontramos a resposta.

Para `nums = [2, 7, 11, 15]` e `target = 9`:

- testamos `2 + 7` e vemos que dá `9`.
- como encontramos a solução, podemos parar imediatamente.

A principal desvantagem é que, no pior caso, vamos examinar muitos pares. Isso faz o custo crescer de forma quadrática.

```text
Two Sum (brute force)

nums   = [2, 7, 11, 15]
target = 9

[1] Estrutura da lista
----------------------

índice:   0    1    2    3
        +----+----+----+----+
valor : | 2  | 7  | 11 | 15 |
        +----+----+----+----+

[2] Execução
------------

Passo 1:
i = 0, j = 0

índice:   0    1    2    3
        +----+----+----+----+
valor : | 2  | 7  | 11 | 15 |
        +----+----+----+----+
          ^i,j

i != j -> não
Ignora este par

Passo 2:
i = 0, j = 1

índice:   0    1    2    3
        +----+----+----+----+
valor : | 2  | 7  | 11 | 15 |
        +----+----+----+----+
          ^i   ^j

i != j -> sim
nums[i] + nums[j] = 2 + 7 = 9
Encontrou a solução

[3] Saída
---------

return [0, 1]
```


### Exemplo de execuçaõ em ASCII sketch

**Exemplo de execuçaõ em ASCII sketch**

Exibição de algum exemplo do passo a passo do algoritmo.


### Implementação da força bruta


In [ ]:
def two_sum_brute_force(nums: list[int], target: int) -> list[int]:
    '''Encontra dois índices cuja soma é igual a `target`, testando todos os pares.

    Descrição:
    ----------
    Esta é a solução mais ingênua para Two Sum. Para cada posição da lista,
    comparamos o valor atual com todos os outros valores até encontrar a soma desejada.

    Parâmetros:
    ----------
    nums : list[int]
        Lista de inteiros que será analisada.
    target : int
        Soma que queremos formar com dois elementos diferentes de `nums`.

    Retorno:
    -------
    list[int]
        Lista com dois índices distintos cuja soma dos valores corresponde a `target`.

    Exceções:
    --------
    ValueError
        Levantada quando não existe solução na lista fornecida.

    Exemplos de uso:
    ----------------
    >>> two_sum_brute_force([2, 7, 11, 15], 9)
    [0, 1]
    >>> two_sum_brute_force([3, 2, 4], 6)
    [1, 2]
    '''
    for i in range(len(nums)):
        for j in range(len(nums)):
            if i != j and nums[i] + nums[j] == target:
                return [i, j]

    raise ValueError("Nenhuma solução encontrada.")


### Testes


In [ ]:
testes = [
    ([2, 7, 11, 15], 9),
    ([3, 2, 4], 6),
    ([3, 3], 6),
    ([1, 4, 5, 6], 10),
    ([-3, 4, 3, 90], 0),
]

for nums, target in testes:
    resposta = two_sum_brute_force(nums, target)
    assert validar_resposta_two_sum(nums, target, resposta), (
        f"Resposta inválida para nums={nums}, target={target}: {resposta}"
    )

print("Todos os testes da Solução 1 passaram.")


## Solução 2 - Melhorias da solução 1


### Descrição da solução

**Descrição da solução**

Agora vamos fazer uma melhoria intermediária. Em vez de comparar cada elemento com **todos** os outros, vamos comparar apenas com os elementos que ainda estão à frente na lista.

Isso mantém a ideia de força bruta, mas evita checar o mesmo par duas vezes. Por exemplo, se já testamos `(i, j)`, não precisamos testar `(j, i)` de novo.

**Como chegamos ao `O(n^2)`?**

Aqui ainda há dois laços aninhados. A diferença é que o laço interno não percorre a lista inteira: ele percorre apenas a parte que falta.

Se a lista tem `n` elementos, o número de comparações no pior caso é:

` (n - 1) + (n - 2) + ... + 2 + 1 `

Essa soma é uma progressão aritmética e resulta em:

` n(n - 1) / 2 `

O termo dominante continua sendo `n^2`, então a ordem assintótica continua `O(n^2)`.

Em outras palavras: esta solução faz menos trabalho que a força bruta ingênua, mas ainda cresce quadraticamente.

**Por que a prática melhora sem mudar a teoria?**

Porque reduzimos redundâncias. Em vez de testar `(i, j)` e `(j, i)`, testamos cada par só uma vez. Isso corta comparações e melhora a constante de tempo, mas não remove o comportamento quadrático.

**Implementação da melhoria**


### Ordem de complexidade (memória e processamento)

**Ordem de complexidade (memória e processamento)**

**Por que isso é uma melhoria?**

- A solução fica mais organizada.
- O número de comparações cai bastante.
- A ordem assintótica continua sendo quadrática, então esta ainda **não** é a solução ótima.

Essa etapa é importante porque mostra uma melhoria real de execução, sem mudar a classe de complexidade.


```text
Two Sum (brute force improved)

nums   = [2, 7, 11, 15]
target = 9

[1] Estrutura da lista
----------------------

índice:   0    1    2    3
        +----+----+----+----+
valor : | 2  | 7  | 11 | 15 |
        +----+----+----+----+

Regra:
j começa em i + 1

[2] Execução
------------

Passo 1:
i = 0, j = 1

índice:   0    1    2    3
        +----+----+----+----+
valor : | 2  | 7  | 11 | 15 |
        +----+----+----+----+
          ^i   ^j

nums[i] + nums[j] = 2 + 7 = 9
Encontrou a solução

[3] Ideia principal
-------------------

Pares percorridos:
(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)

Pares evitados:
(1,0), (2,0), (2,1), (3,0), (3,1), (3,2)

[4] Saída
---------

return [0, 1]
```

- **Tempo:** `O(n^2)`
- **Espaço extra:** `O(1)`


### Mini walkthrough

Exibição de alguns exemplos de como o algoritmo funciona.


### Exemplo de execuçaõ em ASCII sketch

**Exemplo de execuçaõ em ASCII sketch**

Exibição de algum exemplo do passo a passo do algoritmo.


### Implementação da melhoria


In [ ]:
def two_sum_brute_force_improved(nums: list[int], target: int) -> list[int]:
    '''Encontra dois índices cuja soma é igual a `target`, evitando pares repetidos.

    Descrição:
    ----------
    Esta versão ainda é de força bruta, mas faz uma escolha melhor dos pares.
    O índice `i` percorre a lista e o índice `j` olha apenas para a parte
    restante da lista, isto é, posições à frente de `i`.

    Parâmetros:
    ----------
    nums : list[int]
        Lista de inteiros que será analisada.
    target : int
        Soma que queremos formar com dois elementos diferentes de `nums`.

    Retorno:
    -------
    list[int]
        Lista com dois índices distintos cuja soma dos valores corresponde a `target`.

    Exceções:
    --------
    ValueError
        Levantada quando não existe solução na lista fornecida.

    Exemplos de uso:
    ----------------
    >>> two_sum_brute_force_improved([2, 7, 11, 15], 9)
    [0, 1]
    >>> two_sum_brute_force_improved([3, 2, 4], 6)
    [1, 2]
    '''
    for i in range(len(nums) - 1):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]

    raise ValueError("Nenhuma solução encontrada.")


### Testes


Vamos repetir os mesmos testes para confirmar que a melhoria intermediária continua correta.


In [ ]:
for nums, target in testes:
    resposta = two_sum_brute_force_improved(nums, target)
    assert validar_resposta_two_sum(nums, target, resposta), (
        f"Resposta inválida para nums={nums}, target={target}: {resposta}"
    )

print("Todos os testes da Solução 2 passaram.")


## Solução 3 - Melhor solução


### Descrição da solução

**Descrição da solução**

Agora chegamos à solução ótima.

A ideia é guardar, em um dicionário, os números que já apareceram enquanto percorremos a lista. Para cada valor atual `num`, calculamos o complemento necessário:

`complemento = target - num`

Se esse complemento já foi visto antes, encontramos os dois índices.

**Como chegamos ao `O(n)`?**

Há um único laço que percorre a lista uma vez.

Em cada iteração, fazemos duas operações principais:

- calcular o complemento: isso é uma conta simples, então é `O(1)`;
- verificar se o complemento está no dicionário: em tabelas hash bem comportadas, essa busca é `O(1)` amortizado.

Se o custo por elemento é constante e processamos `n` elementos, o custo total é `n * O(1) = O(n)`.

**O que significa `O(1)` amortizado aqui?**

Significa que a operação é constante em média ao longo de várias inserções e buscas, mesmo que ocasionalmente possa haver custos maiores por reestruturação interna da tabela hash.

**Por que o espaço é `O(n)`?**

No pior caso, podemos guardar quase todos os elementos vistos no dicionário. Portanto, a memória cresce linearmente com a entrada.

**Por que esta é a solução preferida em entrevistas?**

Porque ela combina clareza com eficiência. A ideia é simples de explicar e resolve o problema em tempo linear, o que é exatamente o comportamento que queremos para listas grandes.

**Implementação da melhoria**


### Ordem de complexidade (memória e processamento)

**Ordem de complexidade (memória e processamento)**

- **Tempo:** `O(n)` em média / amortizado
- **Espaço extra:** `O(n)`


### Mini walkthrough

**Intuição**

- Percorremos a lista uma única vez.
- Antes de salvar o número atual, verificamos se o complemento já está no dicionário.
- Se estiver, a resposta apareceu naturalmente no caminho.

Para `nums = [2, 7, 11, 15]` e `target = 9`:

- começamos com `2`. O complemento é `7`, que ainda não vimos.
- guardamos `2` no dicionário.
- depois vem `7`. O complemento é `2`, e `2` já foi visto.
- então retornamos os índices `[0, 1]`.

A grande vitória aqui é que a busca pelo complemento passa a ser muito rápida.

```text
Two Sum (hash map)

nums   = [2, 7, 11, 15]
target = 9

[1] Estrutura inicial
---------------------

índice:   0    1    2    3
        +----+----+----+----+
valor : | 2  | 7  | 11 | 15 |
        +----+----+----+----+

vistos = {}

[2] Execução
------------

Passo 1:
indice = 0
numero = 2
complemento = 9 - 2 = 7

7 está em vistos?
não

Ação:
vistos[2] = 0

vistos = {2: 0}

Passo 2:
indice = 1
numero = 7
complemento = 9 - 7 = 2

2 está em vistos?
sim

Resultado:
return [vistos[2], 1]
return [0, 1]

[3] Ideia principal
-------------------

Em vez de testar todos os pares,
o algoritmo guarda os números já vistos.

Para cada número atual:
- calcula o complemento
- procura o complemento no dicionário
- se encontrar, retorna os índices

[4] Saída
---------

return [0, 1]
```


### Exemplo de execuçaõ em ASCII sketch

**Exemplo de execuçaõ em ASCII sketch**

Exibição de algum exemplo do passo a passo do algoritmo.


### Implementação da melhoria


In [ ]:
def two_sum_hash_map(nums: list[int], target: int) -> list[int]:
    '''Encontra dois índices cuja soma é igual a `target` usando um dicionário.

    Descrição:
    ----------
    Esta é a solução ótima clássica para Two Sum. Enquanto percorremos a lista,
    guardamos os valores já vistos em um dicionário que associa cada número ao seu índice.
    Assim, para cada número atual, conseguimos verificar em tempo médio constante se o
    complemento já apareceu.

    Parâmetros:
    ----------
    nums : list[int]
        Lista de inteiros que será analisada.
    target : int
        Soma que queremos formar com dois elementos diferentes de `nums`.

    Retorno:
    -------
    list[int]
        Lista com dois índices distintos cuja soma dos valores corresponde a `target`.

    Exceções:
    --------
    ValueError
        Levantada quando não existe solução na lista fornecida.

    Exemplos de uso:
    ----------------
    >>> two_sum_hash_map([2, 7, 11, 15], 9)
    [0, 1]
    >>> two_sum_hash_map([3, 2, 4], 6)
    [1, 2]
    '''
    vistos: dict[int, int] = {}

    for indice, numero in enumerate(nums):
        complemento = target - numero
        if complemento in vistos:
            return [vistos[complemento], indice]

        vistos[numero] = indice

    raise ValueError("Nenhuma solução encontrada.")


### Testes


Vamos repetir os testes anteriores para confirmar que a solução ótima também está correta.


In [ ]:
for nums, target in testes:
    resposta = two_sum_hash_map(nums, target)
    assert validar_resposta_two_sum(nums, target, resposta), (
        f"Resposta inválida para nums={nums}, target={target}: {resposta}"
    )

print("Todos os testes da Solução 3 passaram.")


## Solução 4 - Solução enxuta para submissão no leetcode


### Descrição da solução

**Descrição da solução**

A solução com `hash map` já é a melhor em termos assintóticos: o tempo continua em `O(n)` e o espaço continua em `O(n)`.

Mesmo assim, ainda dá para brincar com pequenos refinamentos de implementação. A ideia aqui não é mudar a estratégia, nem criar uma nova solução principal. O objetivo é apenas reduzir custo constante.

Essa diferença é importante em plataformas como o LeetCode: um detalhe pode deixar a submissão um pouco mais rápida em uma execução, mas isso não é garantido. O ranking oscila com o ambiente, com o tamanho da entrada e com o ruído de medição.

**Comparação prática**

Agora vamos medir a versão base e a versão otimizada em entradas crescentes. O objetivo aqui é observar tendências, não provar nada de forma formal.

Como ambos os métodos seguem a mesma ideia algorítmica, esperamos curvas muito parecidas. A versão otimizada pode aparecer levemente abaixo em alguns tamanhos, mas a diferença tende a ser pequena. Em benchmarks curtos, o ruído pode inverter a ordem entre uma execução e outra.

**Interpretação dos resultados**

As duas curvas devem crescer como `O(n)`, porque a estratégia algorítmica continua a mesma. A versão otimizada pode ficar um pouco abaixo da base, já que faz menos trabalho por iteração.

Ainda assim, a diferença costuma ser pequena. Quando os resultados ficam muito próximos, isso é esperado: estamos mexendo apenas no custo constante, não na ordem de crescimento.

Em outras palavras: a micro-otimização pode ajudar em alguns casos, mas ela não transforma a solução em algo fundamentalmente melhor do ponto de vista assintótico.

**Implementação da melhoria**

Vamos verificar se a micro-otimização continua produzindo respostas corretas nos mesmos cenários usados nas outras soluções.


### Ordem de complexidade (memória e processamento)

**Ordem de complexidade (memória e processamento)**

**O que foi otimizado**

A solução base faz duas consultas conceituais ao dicionário por iteração:

- primeiro verifica se o complemento existe;
- depois recupera o índice associado.

A versão otimizada usa `get`, que faz essa busca em uma única consulta. Isso não muda a complexidade assintótica, mas pode reduzir um pouco o custo constante.

**O que não mudou**

- a estratégia continua sendo a mesma;
- a estrutura de dados continua sendo o dicionário;
- a complexidade continua `O(n)` em tempo e `O(n)` em espaço.


### Mini walkthrough

Exibição de alguns exemplos de como o algoritmo funciona.


### Exemplo de execuçaõ em ASCII sketch

**Exemplo de execuçaõ em ASCII sketch**

Exibição de algum exemplo do passo a passo do algoritmo.


### Implementação da melhoria


In [ ]:
def two_sum_hash_map_otimizada(nums: list[int], target: int) -> list[int]:
    """Encontra dois índices cuja soma é igual a `target` usando um dicionário,
    com micro-otimizações para reduzir o custo constante da implementação.

    Descrição:
    ----------
    Esta versão mantém a mesma ideia da solução ótima com hash map:
    percorrer a lista apenas uma vez e armazenar os valores já vistos.
    A principal diferença é que ela tenta reduzir pequenas sobrecargas
    de execução usando apenas uma busca no dicionário por iteração.

    Em vez de fazer:
    - `if complemento in vistos`
    - e depois `vistos[complemento]`

    esta implementação usa:
    - `vistos.get(complemento)`

    Assim, evitamos duas consultas separadas ao dicionário.

    Parâmetros:
    ----------
    nums : list[int]
        Lista de inteiros que será analisada.
    target : int
        Soma que queremos formar com dois elementos diferentes de `nums`.

    Retorno:
    -------
    list[int]
        Lista com dois índices distintos cuja soma dos valores corresponde a `target`.

    Exceções:
    --------
    ValueError
        Levantada quando não existe solução na lista fornecida.

    Exemplos de uso:
    ----------------
    >>> two_sum_hash_map_otimizada([2, 7, 11, 15], 9)
    [0, 1]
    >>> two_sum_hash_map_otimizada([3, 2, 4], 6)
    [1, 2]
    """
    vistos: dict[int, int] = {}

    for indice, numero in enumerate(nums):
        indice_anterior = vistos.get(target - numero)
        if indice_anterior is not None:
            return [indice_anterior, indice]

        vistos[numero] = indice

    raise ValueError("Nenhuma solução encontrada.")


In [ ]:
tamanhos_hash_map = [10, 50, 100, 200, 400, 800, 1000]
repeticoes_hash_map = 10

benchmark_hash_map = pd.concat(
    [
        executar_benchmark(two_sum_hash_map, "Hash map base", tamanhos_hash_map, repeticoes_hash_map),
        executar_benchmark(two_sum_hash_map_otimizada, "Hash map otimizado", tamanhos_hash_map, repeticoes_hash_map),
    ],
    ignore_index=True,
)

benchmark_hash_map


In [ ]:
resumo_hash_map = (
    benchmark_hash_map
    .groupby(["tamanho_entrada", "solucao"], as_index=False)
    .agg(tempo_medio_s=("tempo_execucao_s", "mean"), tempo_medio_ms=("tempo_execucao_ms", "mean"))
)

fig, ax = plt.subplots(figsize=(11, 6))
sns.lineplot(
    data=resumo_hash_map,
    x="tamanho_entrada",
    y="tempo_medio_ms",
    hue="solucao",
    marker="o",
    linewidth=2.5,
    ax=ax,
)
ax.set_title("Hash map base versus otimizado")
ax.set_xlabel("Tamanho da entrada (n)")
ax.set_ylabel("Tempo médio de execução (ms)")
plt.tight_layout()
plt.show()


### Testes


In [ ]:
for nums, target in testes:
    resposta = two_sum_hash_map_otimizada(nums, target)
    assert validar_resposta_two_sum(nums, target, resposta), (
        f"Resposta inválida para nums={nums}, target={target}: {resposta}"
    )

print("Todos os testes da versão otimizada passaram.")


## Benchmark

Construção de casos de testes para progressivos para mostrar
que as soluções se tornam cada vez mais otimizadas.
Exibir gráficos e tabelas de comparação entre as soluções pertinentes.

Construção de casos de testes para progressivos para mostrar
que as soluções se tornam cada vez mais otimizadas.
Exibir gráficos e tabelas de comparação entre as soluções pertinentes.

Agora vamos olhar para o comportamento real das implementações.

A ideia aqui é didática: mediremos o tempo de execução em entradas crescentes, repetiremos cada medição algumas vezes e organizaremos os dados em um `DataFrame`.

**Cuidados metodológicos**

- medições práticas possuem ruído;
- um benchmark simples não prova formalmente a complexidade;
- mesmo assim, ele ajuda bastante a enxergar tendências de crescimento;
- em tamanhos pequenos, uma implementação pode parecer mais rápida por causa de constantes, cache e ruído de medição.

Vamos usar a própria implementação das soluções, sem micro-otimizações artificiais.

**Interpretação dos resultados**

O gráfico comparando as três soluções deve mostrar um padrão bem consistente:

- a força bruta cresce mais rápido;
- a melhoria intermediária continua quadrática, mas com uma constante menor;
- a solução com `hash map` cresce de forma aproximadamente linear e tende a se destacar conforme `n` aumenta.

Já o gráfico observado versus teórico ajuda a visualizar que a curva real acompanha a forma prevista pela análise assintótica.

Mesmo com ruído de medição, isso faz sentido porque o que estamos comparando não são valores exatos, e sim o formato geral de crescimento.

Em listas pequenas, uma solução quadrática pode até parecer aceitável. Mas conforme `n` cresce, a diferença entre `O(n^2)` e `O(n)` fica muito mais evidente.

**Comparação entre as abordagens**

| Solução | Ideia central | Tempo | Espaço | Como a complexidade foi derivada | Comportamento prático observado | Vantagens | Desvantagens |
|---|---|---:|---:|---|---|---|---|
| Força bruta | Testar todos os pares possíveis | `O(n^2)` | `O(1)` | Dois laços aninhados; no pior caso, aproximadamente `n(n-1)` comparações | Cresce mais rápido e piora conforme a entrada aumenta | Muito simples de entender | Lenta para listas maiores |
| Melhoria intermediária | Testar apenas a parte restante da lista | `O(n^2)` | `O(1)` | Soma `1 + 2 + ... + (n-1) = n(n-1)/2` | Melhor que a força bruta ingênua, mas ainda quadrática | Reduz redundância e melhora a constante | Continua crescendo quadraticamente |
| Hash map | Guardar números já vistos e procurar o complemento | `O(n)` em média | `O(n)` | Um laço; cada busca/inserção no dicionário é `O(1)` amortizado | Tendência linear e melhor resultado nos tamanhos maiores | Rápida, clara e prática | Usa memória extra |

**Qual usar?**

- **Em entrevistas:** a solução com `hash map` costuma ser a resposta esperada, mas vale mostrar a evolução do raciocínio a partir da força bruta.
- **Em produção:** a solução com `dicionário` é a escolha natural, porque equilibra simplicidade e desempenho.

Observação importante: a solução 3 tem uma versão base, mais didática, e uma versão com micro-otimização. As duas continuam ótimas em termos assintóticos; a variante otimizada tenta apenas melhorar o custo constante.

Se a meta for estudar análise de algoritmos, entender as três abordagens é excelente. Se a meta for resolver o problema da forma mais eficiente, a terceira solução é a ideal.


In [ ]:
tamanhos_benchmark = [10, 50, 100, 200, 400, 800, 1000]
repeticoes_benchmark = 7

resultados_benchmark = pd.concat(
    [
        executar_benchmark(two_sum_brute_force, "Força bruta", tamanhos_benchmark, repeticoes_benchmark),
        executar_benchmark(two_sum_brute_force_improved, "Melhoria intermediária", tamanhos_benchmark, repeticoes_benchmark),
        executar_benchmark(two_sum_hash_map, "Hash map", tamanhos_benchmark, repeticoes_benchmark),
    ],
    ignore_index=True,
)

resultados_benchmark


In [ ]:
resumo_benchmark = (
    resultados_benchmark
    .groupby(["tamanho_entrada", "solucao"], as_index=False)
    .agg(
        tempo_medio_s=("tempo_execucao_s", "mean"),
        tempo_desvio_s=("tempo_execucao_s", "std"),
        tempo_medio_ms=("tempo_execucao_ms", "mean"),
    )
)

resumo_benchmark


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.lineplot(
    data=resumo_benchmark,
    x="tamanho_entrada",
    y="tempo_medio_ms",
    hue="solucao",
    marker="o",
    linewidth=2.5,
    ax=ax,
)
ax.set_title("Tempo médio observado por solução")
ax.set_xlabel("Tamanho da entrada (n)")
ax.set_ylabel("Tempo médio de execução (ms)")
plt.tight_layout()
plt.show()


In [ ]:
def construir_curva_teorica_ajustada(df: pd.DataFrame, coluna_teorica: str) -> pd.DataFrame:
    '''Cria uma curva teórica apenas para comparação visual com a curva observada.

    A ideia é escalar a curva teórica para que ela fique na mesma ordem de grandeza
    dos tempos medidos, sem afirmar que os valores absolutos são iguais.
    '''
    primeiro = df.iloc[0]
    fator = primeiro["tempo_medio_s"] / primeiro[coluna_teorica]
    saida = df.copy()
    saida["tempo_teorico_ajustado_ms"] = saida[coluna_teorica] * fator * 1000
    return saida


comparacao_curvas = []
for solucao, coluna_teorica in [
    ("Força bruta", "teoria_n2"),
    ("Melhoria intermediária", "teoria_n2_metade"),
    ("Hash map", "teoria_n"),
]:
    sub = resumo_benchmark[resumo_benchmark["solucao"] == solucao].copy()
    sub["teoria_n"] = sub["tamanho_entrada"]
    sub["teoria_n2"] = sub["tamanho_entrada"] ** 2
    sub["teoria_n2_metade"] = (sub["tamanho_entrada"] * (sub["tamanho_entrada"] - 1)) / 2
    sub = construir_curva_teorica_ajustada(sub, coluna_teorica)
    sub["solucao"] = solucao
    comparacao_curvas.append(sub)

comparacao_curvas = pd.concat(comparacao_curvas, ignore_index=True)
comparacao_curvas


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

configuracoes = [
    ("Força bruta", "teoria_n2", "Crescimento observado versus teórico - Força bruta"),
    ("Melhoria intermediária", "teoria_n2_metade", "Crescimento observado versus teórico - Melhoria intermediária"),
    ("Hash map", "teoria_n", "Crescimento observado versus teórico - Hash map"),
]

for ax, (solucao, coluna_teorica, titulo) in zip(axes, configuracoes):
    sub = comparacao_curvas[comparacao_curvas["solucao"] == solucao]
    sns.lineplot(data=sub, x="tamanho_entrada", y="tempo_medio_ms", marker="o", linewidth=2.5, label="Observado", ax=ax)
    sns.lineplot(data=sub, x="tamanho_entrada", y="tempo_teorico_ajustado_ms", marker="o", linewidth=2.5, linestyle="--", label="Teórico ajustado", ax=ax)
    ax.set_title(titulo)
    ax.set_xlabel("n")
    ax.set_ylabel("Tempo (ms)")
    ax.legend()

plt.tight_layout()
plt.show()
